In [ ]:
!pip uninstall -y \
    datasets \
    transformers \
    sentence-transformers \
    huggingface_hub \
    tokenizers \
    accelerate \
    peft

!pip install -q --no-cache-dir \
    "huggingface_hub==0.34.4" \
    "transformers==4.55.0" \
    "datasets==4.0.0" \
    "sentence-transformers==5.1.0" \
    faiss-cpu \
    rank-bm25

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: sentence-transformers 5.6.0
Uninstalling sentence-transformers-5.6.0:
  Successfully uninstalled sentence-transformers-5.6.0
Found existing installation: huggingface_hub 1.23.0
Uninstalling huggingface_hub-1.23.0:
  Successfully uninstalled huggingface_hub-1.23.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 443.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import datasets
import huggingface_hub
import transformers
import sentence_transformers

print("datasets:", datasets.__version__)
print("hub:", huggingface_hub.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

datasets: 4.0.0
hub: 0.34.4
transformers: 4.55.0
sentence-transformers: 5.1.0


In [ ]:
# Install once per runtime
!pip -q install datasets rank_bm25 sentence-transformers faiss-cpu scikit-learn tqdm

# Optional LLM clients (install both; use whichever key you have)
!pip -q install groq google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 13.3 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

raw = load_dataset("pubmed_qa", "pqa_labeled")

README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
import os, re, json, time, random
from typing import List, Dict, Any, Tuple, Optional
from collections import Counter

import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss

LLM_PROVIDER = "groq"
GROQ_API_KEY = "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"
GEMINI_API_KEY = "AQ.Ab8RN6KAsJcXGek4bQGX73tR1JkJjUatxShTcDNgzguX66W1sw"
GROQ_MODEL = "llama-3.1-8b-instant"
GEMINI_MODEL = "gemini-2.0-flash"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K_BM25 = 8
TOP_K_DENSE = 8
TOP_K_FINAL = 5
SEED = 42
EVAL_N = 40
DEMO_N = 5

random.seed(SEED)
np.random.seed(SEED)


try:
    from google.colab import userdata
    GROQ_API_KEY = GROQ_API_KEY or userdata.get("GROQ_API_KEY")
    GEMINI_API_KEY = GEMINI_API_KEY or userdata.get("GEMINI_API_KEY")
except Exception:
    GROQ_API_KEY = GROQ_API_KEY or os.environ.get("GROQ_API_KEY", "")
    GEMINI_API_KEY = GEMINI_API_KEY or os.environ.get("GEMINI_API_KEY", "")

print("Provider:", LLM_PROVIDER)
print("Groq key set:", bool(GROQ_API_KEY))
print("Gemini key set:", bool(GEMINI_API_KEY))

Provider: groq
Groq key set: True
Gemini key set: True


In [ ]:

raw = load_dataset("pubmed_qa", "pqa_labeled")
print(raw)


all_rows = list(raw["train"])
random.shuffle(all_rows)

split_at = int(0.8 * len(all_rows))
train_rows = all_rows[:split_at]
val_rows = all_rows[split_at:]
print(f"train={len(train_rows)} val={len(val_rows)}")

ex = train_rows[0]
print("\nExample question:", ex["question"])
print("Decision:", ex["final_decision"])
print("Long answer:", (ex.get("long_answer") or "")[:300], "...")
print("Num contexts:", len(ex["context"]["contexts"]))

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})
train=800 val=200

Example question: Does open access endoscopy close the door to an adequately informed patient?
Decision: yes
Long answer: Patients undergoing open access procedures are less likely to be properly informed about their endoscopic procedure. Our results indicate that with open access endoscopy, a defined mechanism needs to be in place for communication of endoscopic results to the patient. ...
Num contexts: 3


In [ ]:
def normalize_decision(x: str) -> str:
    x = (x or "").strip().lower()
    if x in {"yes", "no", "maybe"}:
        return x
    return x

def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", (text or "").lower())

docs = []
for qa_idx, row in enumerate(train_rows):
    ctxs = row["context"]["contexts"]
    for ctx_i, passage in enumerate(ctxs):
        docs.append({
            "doc_id": f"train-{qa_idx}-{ctx_i}",
            "text": passage.strip(),
            "qa_idx": qa_idx,
            "ctx_i": ctx_i,
            "question": row["question"],
            "gold": normalize_decision(row["final_decision"]),
        })

print("Indexed passages:", len(docs))
print("Sample passage:\n", docs[0]["text"][:400], "...")

Indexed passages: 2680
Sample passage:
 The use of open access endoscopy is increasing. Its effect on the adequacy of patient informed consent, procedure acceptance and the impact on subsequent communication/transfer of procedure results to the patient have not been evaluated. The aim of our study was to compare the extent of preknowledge of procedures and test explanation, patient medical complexity, information transfer and overall pa ...


In [ ]:
tokenized_corpus = [tokenize(d["text"]) for d in docs]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query: str, top_k: int = TOP_K_BM25) -> List[Tuple[int, float]]:
    scores = bm25.get_scores(tokenize(query))
    idxs = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in idxs if scores[i] > 0]

q = train_rows[0]["question"]
hits = bm25_search(q, top_k=3)
print("Query:", q)
for i, s in hits:
    print("\nscore={:.3f} | {}...".format(s, docs[i]["text"][:220]))

Query: Does open access endoscopy close the door to an adequately informed patient?

score=33.446 | The use of open access endoscopy is increasing. Its effect on the adequacy of patient informed consent, procedure acceptance and the impact on subsequent communication/transfer of procedure results to the patient have no...

score=21.019 | Some studies suggest that open access articles are more often cited than non-open access articles. However, the relationship between open access and citations count in a discipline such as intensive care medicine has not...

score=17.833 | The median number (interquartile range) of citations of non-open access articles was 8 (4-12) versus 9 (6-18) in the case of open access articles (p=0.084). In the highest citation range (>8), the citation count was 13 (...


In [ ]:
print("Loading embedder:", EMBED_MODEL)
embedder = SentenceTransformer(EMBED_MODEL)

BATCH = 64
texts = [d["text"] for d in docs]
embs = embedder.encode(
    texts, batch_size=BATCH, show_progress_bar=True, normalize_embeddings=True
)
embs = np.asarray(embs, dtype=np.float32)

dim = embs.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine via normalized vectors
index.add(embs)
print("FAISS vectors:", index.ntotal)

def dense_search(query: str, top_k: int = TOP_K_DENSE) -> List[Tuple[int, float]]:
    qv = embedder.encode([query], normalize_embeddings=True)
    qv = np.asarray(qv, dtype=np.float32)
    scores, idxs = index.search(qv, top_k)
    out = []
    for i, s in zip(idxs[0], scores[0]):
        if int(i) >= 0:
            out.append((int(i), float(s)))
    return out

def rrf_fuse(
    result_lists: List[List[Tuple[int, float]]], k: int = 60, top_k: int = TOP_K_FINAL
) -> List[Tuple[int, float]]:

    fused: Dict[int, float] = {}
    for results in result_lists:
        for rank, (doc_i, _score) in enumerate(results, start=1):
            fused[doc_i] = fused.get(doc_i, 0.0) + 1.0 / (k + rank)
    ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return ranked

def hybrid_search(
    query: str, mode: str = "hybrid", top_k: int = TOP_K_FINAL
) -> List[Dict[str, Any]]:
    mode = mode.lower()
    bm = bm25_search(query, top_k=TOP_K_BM25)
    de = dense_search(query, top_k=TOP_K_DENSE)

    if mode == "bm25":
        ranked = bm[:top_k]
    elif mode == "dense":
        ranked = de[:top_k]
    else:
        ranked = rrf_fuse([bm, de], top_k=top_k)

    out = []
    for doc_i, score in ranked:
        d = docs[doc_i]
        out.append({
            "doc_id": d["doc_id"],
            "score": score,
            "text": d["text"],
            "qa_idx": d["qa_idx"],
        })
    return out

q = val_rows[0]["question"]
print("VAL question:", q)
for mode in ["bm25", "dense", "hybrid"]:
    hits = hybrid_search(q, mode=mode, top_k=3)
    print("\n=== {} ===".format(mode.upper()))
    for h in hits:
        print("score={:.4f} | {}...".format(h["score"], h["text"][:180]))

Loading embedder: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

FAISS vectors: 2680
VAL question: Does preoperative anemia adversely affect colon and rectal surgery outcomes?

=== BM25 ===
score=20.2139 | The main treatment for rectal carcinoma is surgery. Preoperative chemoradiation (CRT) is advocated to reduce local recurrence and improve resection of mid and low tethered rectal t...
score=17.6676 | The effect of preoperative education on anxiety and postoperative outcomes of cardiac surgery patients remains unclear.AIM: The aim of the study was to estimate the effectiveness o...
score=14.3541 | A total of 66 cases were queried and divided into 2 groups as constipated (n = 35; mean age, 6.8 ± 2.9 years) and control (n = 31; mean age, 8.4 ± 3.8 years) according to Rome III ...

=== DENSE ===
score=0.5561 | This prospective, randomized study was designed to evaluate whether or not early postoperative feeding (claimed as a unique benefit of laparoscopic surgery) is possible after lapar...
score=0.4931 | The quality of surgical excision is held to be

In [ ]:
SYSTEM_PROMPT = """You are a careful biomedical research assistant.
Answer ONLY using the provided evidence passages.
Write a detailed answer (5-10 sentences) explaining the findings.
Cite evidence as [1], [2], etc.
End with exactly one line in this format:
Verdict: Yes
or
Verdict: No
or
Verdict: Maybe
If evidence is insufficient or conflicting, choose Maybe."""

def build_prompt(question: str, passages: List[str]) -> str:
    blocks = []
    for i, p in enumerate(passages, 1):
        blocks.append(f"[{i}] {p}")
    evidence = "\n\n".join(blocks)
    return (
        f"Question: {question}\n\n"
        f"Evidence passages:\n{evidence}\n\n"
        "Write a detailed answer with citations, then the Verdict line."
    )


def call_groq(prompt: str) -> str:
    from groq import Groq
    client = Groq(api_key=GROQ_API_KEY)
    resp = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1,
        max_tokens=700,
    )
    return resp.choices[0].message.content.strip()


def call_gemini(prompt: str) -> str:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel(GEMINI_MODEL, system_instruction=SYSTEM_PROMPT)
    resp = model.generate_content(prompt)
    return (resp.text or "").strip()


def extractive_answer(question: str, passages: List[str]) -> str:

    joined = " ".join(passages)[:1200]
    return (
        f"Based on the retrieved evidence, the literature related to: '{question}'. "
        f"Key evidence excerpts: {joined[:800]}... "
        f"Because this fallback cannot fully reason, the decision is uncertain.\n"
        f"Verdict: Maybe"
    )


def generate_answer(question: str, passages: List[str]) -> str:
    prompt = build_prompt(question, passages)
    if LLM_PROVIDER == "groq":
        if not GROQ_API_KEY:
            print("No GROQ key — using extractive fallback")
            return extractive_answer(question, passages)
        return call_groq(prompt)
    if LLM_PROVIDER == "gemini":
        if not GEMINI_API_KEY:
            print("No Gemini key — using extractive fallback")
            return extractive_answer(question, passages)
        return call_gemini(prompt)
    return extractive_answer(question, passages)


def parse_verdict(text: str) -> Optional[str]:
    m = re.search(r"Verdict:\s*(Yes|No|Maybe)", text, flags=re.I)
    if not m:
        return None
    return m.group(1).lower()


def answer_question(
    question: str, mode: str = "hybrid", top_k: int = TOP_K_FINAL
) -> Dict[str, Any]:
    hits = hybrid_search(question, mode=mode, top_k=top_k)
    passages = [h["text"] for h in hits]
    raw_ans = generate_answer(question, passages)
    verdict = parse_verdict(raw_ans)
    return {
        "question": question,
        "mode": mode,
        "hits": hits,
        "answer": raw_ans,
        "verdict": verdict,
    }

demo_q = val_rows[0]["question"]
demo_gold = normalize_decision(val_rows[0]["final_decision"])
result = answer_question(demo_q, mode="hybrid")
print("Question:", demo_q)
print("Gold:", demo_gold)
print("Pred verdict:", result["verdict"])
print("\n----- MODEL OUTPUT -----\n")
print(result["answer"])
print("\n----- TOP EVIDENCE -----\n")
for i, h in enumerate(result["hits"], 1):
    print(f"[{i}] {h['text'][:260]}...\n")

Question: Does preoperative anemia adversely affect colon and rectal surgery outcomes?
Gold: yes
Pred verdict: maybe

----- MODEL OUTPUT -----

The available evidence does not directly address the impact of preoperative anemia on colon and rectal surgery outcomes. However, studies have investigated various factors that may influence surgical outcomes, such as the quality of surgical excision [4], the effect of preoperative education on anxiety and postoperative outcomes [3], and the feasibility of early postoperative feeding after laparotomy and colorectal resection [2]. These studies suggest that factors other than preoperative anemia may play a significant role in determining surgical outcomes.

Regarding preoperative anemia specifically, there is limited information available. However, preoperative anemia has been identified as a risk factor for adverse outcomes in various surgical procedures, including cardiac surgery [3]. In the context of colon and rectal surgery, preoperative an

In [ ]:
def evaluate(rows, mode="hybrid", n=EVAL_N, sleep_s=0.15):
    rows = rows[:n]
    y_true, y_pred = [], []
    parsed_ok = 0
    examples = []

    for row in tqdm(rows, desc=f"eval-{mode}"):
        q = row["question"]
        gold = normalize_decision(row["final_decision"])
        out = answer_question(q, mode=mode)
        pred = out["verdict"]
        if pred is None:
            pred = "maybe"
        else:
            parsed_ok += 1
        y_true.append(gold)
        y_pred.append(pred)
        examples.append({
            "question": q,
            "gold": gold,
            "pred": pred,
            "answer": out["answer"],
        })
        time.sleep(sleep_s)

    acc = float(np.mean([t == p for t, p in zip(y_true, y_pred)]))
    print(f"\nMode={mode} N={len(rows)}")
    print(f"Verdict accuracy: {acc:.3f}")
    print(f"Parsed Verdict line: {parsed_ok}/{len(rows)}")
    print("Gold distribution:", Counter(y_true))
    print("Pred distribution:", Counter(y_pred))
    return {
        "accuracy": acc,
        "examples": examples,
        "y_true": y_true,
        "y_pred": y_pred,
    }

eval_hybrid = evaluate(val_rows, mode="hybrid", n=EVAL_N)

print("\n===== Sample detailed outputs =====")
for ex in eval_hybrid["examples"][:DEMO_N]:
    print("\nQ:", ex["question"])
    print("Gold:", ex["gold"], "| Pred:", ex["pred"])
    print(ex["answer"][:900])
    print("-" * 60)

eval-hybrid:   0%|          | 0/40 [00:00<?, ?it/s]


Mode=hybrid N=40
Verdict accuracy: 0.075
Parsed Verdict line: 40/40
Gold distribution: Counter({'yes': 20, 'no': 18, 'maybe': 2})
Pred distribution: Counter({'maybe': 38, 'yes': 2})

===== Sample detailed outputs =====

Q: Does preoperative anemia adversely affect colon and rectal surgery outcomes?
Gold: yes | Pred: maybe
The available evidence does not directly address the question of whether preoperative anemia adversely affects colon and rectal surgery outcomes. However, studies have investigated the impact of preoperative factors on surgical outcomes. For instance, the quality of surgical excision is a major determinant of outcome following surgery for rectal cancer [4]. Additionally, preoperative education has been shown to have a positive impact on anxiety and postoperative outcomes in cardiac surgery patients [3]. 

In contrast, studies on preoperative chemoradiation (CRT) for rectal carcinoma [1] and early postoperative feeding after laparotomy and colorectal resection [2] do 

In [ ]:

RUN_ABLATION = False
ABLATION_N = 15

if RUN_ABLATION:
    eval_bm25 = evaluate(val_rows, mode="bm25", n=ABLATION_N)
    eval_hyb2 = evaluate(val_rows, mode="hybrid", n=ABLATION_N)
    print("\nABLATION")
    print("BM25 accuracy:", round(eval_bm25["accuracy"], 3))
    print("Hybrid accuracy:", round(eval_hyb2["accuracy"], 3))
else:
    print("Skipped ablation. Set RUN_ABLATION=True to compare BM25 vs Hybrid.")

Skipped ablation. Set RUN_ABLATION=True to compare BM25 vs Hybrid.


In [ ]:
payload = {
    "provider": LLM_PROVIDER,
    "embed_model": EMBED_MODEL,
    "eval_n": EVAL_N,
    "hybrid_accuracy": eval_hybrid["accuracy"],
    "demos": eval_hybrid["examples"][:DEMO_N],
}
with open("pubmedqa_day3_results.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)


In [ ]:
# ========= FAIR EVAL (use each question's own evidence) =========
def answer_with_given_passages(question, passages):
    raw = generate_answer(question, passages[:TOP_K_FINAL])
    return raw, parse_verdict(raw)

def evaluate_oracle_contexts(rows, n=40, sleep_s=0.15):
    rows = rows[:n]
    y_true, y_pred = [], []
    examples = []

    for row in tqdm(rows, desc="oracle-eval"):
        q = row["question"]
        gold = normalize_decision(row["final_decision"])
        passages = row["context"]["contexts"]
        raw, pred = answer_with_given_passages(q, passages)
        if pred is None:
            pred = "maybe"
        y_true.append(gold)
        y_pred.append(pred)
        examples.append({"question": q, "gold": gold, "pred": pred, "answer": raw})
        time.sleep(sleep_s)

    acc = float(np.mean([t == p for t, p in zip(y_true, y_pred)]))
    print(f"Oracle-context accuracy: {acc:.3f}")
    print("Gold:", Counter(y_true))
    print("Pred:", Counter(y_pred))
    return {"accuracy": acc, "examples": examples}

oracle = evaluate_oracle_contexts(val_rows, n=20)

oracle-eval:   0%|          | 0/20 [00:00<?, ?it/s]

Oracle-context accuracy: 0.600
Gold: Counter({'yes': 9, 'no': 9, 'maybe': 2})
Pred: Counter({'yes': 7, 'maybe': 7, 'no': 6})


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

q = widgets.Textarea(
    value="",
    placeholder="Type a Yes/No biomedical question...",
    layout=widgets.Layout(width="100%", height="90px"),
)
mode = widgets.Dropdown(options=["hybrid", "bm25", "dense"], value="hybrid", description="Mode:")
btn = widgets.Button(description="Get answer", button_style="success")
out = widgets.Output()

def on_click(_):
    with out:
        clear_output()
        if not q.value.strip():
            print("Please type a question.")
            return
        result = answer_question(q.value.strip(), mode=mode.value)
        print("VERDICT:", result["verdict"])
        print("\n----- ANSWER -----\n")
        print(result["answer"])
        print("\n----- EVIDENCE -----\n")
        for i, h in enumerate(result["hits"], 1):
            print(f"[{i}] {h['text'][:450]}...\n")

btn.on_click(on_click)
display(q, mode, btn, out)

Textarea(value='', layout=Layout(height='90px', width='100%'), placeholder='Type a Yes/No biomedical question.…

Dropdown(description='Mode:', options=('hybrid', 'bm25', 'dense'), value='hybrid')

Button(button_style='success', description='Get answer', style=ButtonStyle())

Output()